# ACC Analysis — Current Range-Sensing CTG-PD Benchmark

This notebook is the analysis stage of the active pipeline. Experiment settings are loaded from `experiment_manifest_acc_range_pd_norm_v3.json` through `core.config`, so the notebook, optimizer runner, and manuscript-facing analysis use the same contract.

**Active experiment contract**
- Controller gains: `[k_v, k_p, k_d]`
- Search bounds: `[-100, 100]^3`
- Sensing boundary: simulator-side range → configurable range sensor → finite-difference range-rate estimate
- Nominal sensor noise/bias: disabled (`0.0`), but the simulator accepts seeded range noise without changing the controller API
- Objectives: normalized IAE, ISE, ITAE, and ITSE
- Optimizers: GA, DE, FVRADE, RNEGDE, Neighborhood-SHADE, and DE-NPS-FP
- 30 runs × 30 iterations × population 100 for each optimizer-objective pair
- Function-evaluation counts and runtime are analyzed explicitly
- Closed-loop validation uses the observed run closest to the **median final objective**, not the stochastic best run

Only checkpoints whose filename matches the active benchmark and optimizer registry are loaded.


## 0. Configuration and experiment contract


In [ ]:
from __future__ import annotations

import itertools
import math
import pickle
from pathlib import Path
from typing import List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import friedmanchisquare, rankdata, shapiro, wilcoxon

from core.config import load_manifest

MANIFEST = load_manifest()
BENCHMARK_KEY = MANIFEST["benchmark"]
CHECKPOINT_DIR = Path("checkpoint_jobs")
DATA_DIR = Path("data")

OBJECTIVES = list(MANIFEST["optimization"]["objectives"])
ALGORITHMS = list(MANIFEST["optimization"]["algorithms"])
ALGO_LABELS = dict(MANIFEST["optimization"]["algorithm_display_names"])
GAIN_NAMES = list(MANIFEST["controller"]["decision_vector"])

EXPECTED_RUNS = int(MANIFEST["optimization"]["independent_runs"])
EXPECTED_ITERATIONS = int(MANIFEST["optimization"]["max_iterations"])
POPULATION_SIZE = int(MANIFEST["optimization"]["population_size"])
MIN_PARAMS = np.asarray(MANIFEST["controller"]["min_bounds"], dtype=float)
MAX_PARAMS = np.asarray(MANIFEST["controller"]["max_bounds"], dtype=float)

DT = 1.0 / float(MANIFEST["execution_backend"]["sampling_hz"])
VEHICLE_MASS = float(MANIFEST["physics"]["nominal_mass_kg"])
C_R_NOMINAL = float(MANIFEST["physics"]["rolling_resistance"])
C_R_PLUS_20 = C_R_NOMINAL * 1.20
TIME_GAP = float(MANIFEST["controller"]["time_gap_s"])
STANDSTILL_DISTANCE = float(MANIFEST["controller"]["standstill_distance_m"])
SPEED_ERROR_SCALE = float(MANIFEST["controller"]["speed_reference_m_per_s"])
SPACING_ERROR_SCALE = STANDSTILL_DISTANCE + TIME_GAP * SPEED_ERROR_SCALE
TTC_THRESHOLD = 1.5

NOMINAL_SENSOR = {
    "sensor_noise_std_m": float(MANIFEST["sensing"]["nominal_noise_std_m"]),
    "sensor_bias_m": float(MANIFEST["sensing"]["nominal_bias_m"]),
    "sensor_seed": int(MANIFEST["sensing"]["nominal_noise_seed"]),
    "sensor_min_distance_m": float(MANIFEST["sensing"]["nominal_min_distance_m"]),
}

PARAM_SELECTION = "median"

for subdir in [
    "aggregated",
    "statistics",
    "statistics/normality/qqplots",
    "statistics/comparative",
    "convergence/summary",
    "time_response/cruise_control",
    "time_response/car_following/ttc",
    "time_response/car_following/motion",
    "time_response/car_following/sensing",
    "robustness/cruise_control/results",
    "robustness/cruise_control/scenarios",
    "robustness/car_following/results",
    "robustness/car_following/scenarios",
]:
    (DATA_DIR / subdir).mkdir(parents=True, exist_ok=True)


def algo_label(name: str) -> str:
    return ALGO_LABELS.get(name, name.upper())


print("Benchmark:", BENCHMARK_KEY)
print("Algorithms:", [algo_label(a) for a in ALGORITHMS])
print("Objectives:", [x.upper() for x in OBJECTIVES])
print("Runs / iterations / population:", EXPECTED_RUNS, EXPECTED_ITERATIONS, POPULATION_SIZE)
print("Gain bounds:", list(zip(GAIN_NAMES, MIN_PARAMS, MAX_PARAMS)))
print("Nominal sensor:", NOMINAL_SENSOR)


## 1. Load and aggregate run artifacts

Only checkpoint files from the active benchmark are accepted. The filename contract is `<benchmark>_run_<run>_<algorithm>_<objective>.pkl`; algorithm and objective names must match the manifest exactly.


In [ ]:
def parse_checkpoint_filename(filename: str):
    prefix = f"{BENCHMARK_KEY}_run_"
    if not (filename.startswith(prefix) and filename.endswith(".pkl")):
        return None

    body = filename[len(prefix):-4]
    first_sep = body.find("_")
    if first_sep < 0:
        return None

    try:
        run_id = int(body[:first_sep])
    except ValueError:
        return None

    rest = body[first_sep + 1:]
    for obj in sorted(OBJECTIVES, key=len, reverse=True):
        suffix = f"_{obj}"
        if rest.endswith(suffix):
            algo = rest[:-len(suffix)]
            if algo in ALGORITHMS:
                return run_id, algo, obj
    return None


def load_checkpoints():
    rows = []
    if not CHECKPOINT_DIR.exists():
        raise FileNotFoundError(f"Missing checkpoint directory: {CHECKPOINT_DIR}")

    for path in sorted(CHECKPOINT_DIR.glob("*.pkl")):
        parsed = parse_checkpoint_filename(path.name)
        if parsed is None:
            continue
        run_id, algo, obj = parsed
        with path.open("rb") as f:
            payload = pickle.load(f)

        rows.append({
            "run_id": run_id,
            "algo": algo,
            "obj": obj,
            "seed": payload.get("seed"),
            "best_fitness": payload.get("best_fitness"),
            "best_solution": payload.get("best_solution"),
            "curve": payload.get("curve"),
            "duration": payload.get("duration"),
            "function_evaluations": payload.get("function_evaluations"),
            "source_file": path.name,
        })

    rows.sort(key=lambda x: (x["obj"], x["algo"], x["run_id"]))
    print(f"Loaded {len(rows)} checkpoint(s) for {BENCHMARK_KEY}.")
    return rows


checkpoint_rows = load_checkpoints()


In [ ]:
def aggregate_checkpoints(rows):
    recaps = {
        obj: {
            algo: {
                "run_ids": [],
                "seeds": [],
                "best_fitness": [],
                "best_solutions": [],
                "curves": [],
                "durations": [],
                "function_evaluations": [],
                "source_files": [],
            }
            for algo in ALGORITHMS
        }
        for obj in OBJECTIVES
    }

    for row in rows:
        bucket = recaps[row["obj"]][row["algo"]]
        bucket["run_ids"].append(row["run_id"])
        bucket["seeds"].append(row["seed"])
        bucket["best_fitness"].append(row["best_fitness"])
        bucket["best_solutions"].append(row["best_solution"])
        bucket["curves"].append(row["curve"])
        bucket["durations"].append(row["duration"])
        bucket["function_evaluations"].append(row["function_evaluations"])
        bucket["source_files"].append(row["source_file"])

    return recaps


recaps = aggregate_checkpoints(checkpoint_rows)

aggregated_payloads = {
    "all_best_fitnesses.pkl": {
        obj: {algo: recaps[obj][algo]["best_fitness"] for algo in ALGORITHMS}
        for obj in OBJECTIVES
    },
    "all_best_solutions.pkl": {
        obj: {algo: recaps[obj][algo]["best_solutions"] for algo in ALGORITHMS}
        for obj in OBJECTIVES
    },
    "all_curves.pkl": {
        obj: {algo: recaps[obj][algo]["curves"] for algo in ALGORITHMS}
        for obj in OBJECTIVES
    },
    "all_durations.pkl": {
        obj: {algo: recaps[obj][algo]["durations"] for algo in ALGORITHMS}
        for obj in OBJECTIVES
    },
    "all_function_evaluations.pkl": {
        obj: {algo: recaps[obj][algo]["function_evaluations"] for algo in ALGORITHMS}
        for obj in OBJECTIVES
    },
    "all_run_ids.pkl": {
        obj: {algo: recaps[obj][algo]["run_ids"] for algo in ALGORITHMS}
        for obj in OBJECTIVES
    },
    "all_seeds.pkl": {
        obj: {algo: recaps[obj][algo]["seeds"] for algo in ALGORITHMS}
        for obj in OBJECTIVES
    },
}

for filename, payload in aggregated_payloads.items():
    with (DATA_DIR / "aggregated" / filename).open("wb") as f:
        pickle.dump(payload, f)

coverage_rows = []
for obj in OBJECTIVES:
    for algo in ALGORITHMS:
        run_ids = recaps[obj][algo]["run_ids"]
        coverage_rows.append({
            "Objective": obj.upper(),
            "Algorithm": algo_label(algo),
            "CompletedRuns": len(run_ids),
            "ExpectedRuns": EXPECTED_RUNS,
            "MissingRuns": EXPECTED_RUNS - len(run_ids),
            "RunIDs": ",".join(map(str, run_ids)),
        })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(DATA_DIR / "aggregated" / "run_coverage.csv", index=False)
coverage_df


## 2. Data Summarization


In [ ]:
def _finite_float_array(values):
    arr = np.asarray([v for v in values if v is not None], dtype=float)
    return arr[np.isfinite(arr)]


def generate_statistical_report(recaps):
    rows = []

    for obj in OBJECTIVES:
        for algo in ALGORITHMS:
            bucket = recaps[obj][algo]
            fitness = _finite_float_array(bucket["best_fitness"])
            if fitness.size == 0:
                continue

            solutions = np.asarray(bucket["best_solutions"], dtype=float)
            durations = _finite_float_array(bucket["durations"])
            fe = _finite_float_array(bucket["function_evaluations"])

            best_idx = int(np.nanargmin(np.asarray(bucket["best_fitness"], dtype=float)))
            worst_idx = int(np.nanargmax(np.asarray(bucket["best_fitness"], dtype=float)))
            q25, q75 = np.percentile(fitness, [25, 75])
            best_solution = np.asarray(bucket["best_solutions"][best_idx], dtype=float)
            worst_solution = np.asarray(bucket["best_solutions"][worst_idx], dtype=float)

            row = {
                "Objective": obj.upper(),
                "Algorithm": algo_label(algo),
                "Runs": int(fitness.size),
                "Best Fitness": float(np.min(fitness)),
                "Worst Fitness": float(np.max(fitness)),
                "Mean Fitness": float(np.mean(fitness)),
                "Median Fitness": float(np.median(fitness)),
                "Std Dev Fitness": float(np.std(fitness, ddof=1)) if fitness.size > 1 else 0.0,
                "IQR Fitness": float(q75 - q25),
                "Mean Duration (s)": float(np.mean(durations)) if durations.size else np.nan,
                "Std Duration (s)": float(np.std(durations, ddof=1)) if durations.size > 1 else 0.0,
                "Mean FE": float(np.mean(fe)) if fe.size else np.nan,
                "Median FE": float(np.median(fe)) if fe.size else np.nan,
                "Min FE": float(np.min(fe)) if fe.size else np.nan,
                "Max FE": float(np.max(fe)) if fe.size else np.nan,
            }

            for idx, gain in enumerate(GAIN_NAMES):
                row[f"Best {gain}"] = float(best_solution[idx])
                row[f"Worst {gain}"] = float(worst_solution[idx])
                row[f"Mean {gain}"] = float(np.mean(solutions[:, idx]))
                row[f"Std {gain}"] = float(np.std(solutions[:, idx], ddof=1)) if len(solutions) > 1 else 0.0

            rows.append(row)

    df = pd.DataFrame(rows)
    csv_path = DATA_DIR / "statistics" / "statistical_report.csv"
    txt_path = DATA_DIR / "statistics" / "statistical_report.txt"
    df.to_csv(csv_path, index=False)

    with txt_path.open("w", encoding="utf-8") as f:
        f.write(f"Statistical Report — {BENCHMARK_KEY}\n")
        f.write("=" * 72 + "\n")
        f.write(f"Gain vector: {GAIN_NAMES}\n")
        f.write(f"Bounds: {MIN_PARAMS.tolist()} to {MAX_PARAMS.tolist()}\n")
        f.write(f"Expected runs per pair: {EXPECTED_RUNS}\n\n")
        for obj in OBJECTIVES:
            f.write(f"\n===== {obj.upper()} =====\n")
            subset = df[df["Objective"] == obj.upper()]
            for _, r in subset.iterrows():
                f.write(
                    f"{r['Algorithm']}: median={r['Median Fitness']:.8g}, "
                    f"IQR={r['IQR Fitness']:.8g}, mean={r['Mean Fitness']:.8g}, "
                    f"std={r['Std Dev Fitness']:.8g}, mean_FE={r['Mean FE']:.1f}, "
                    f"mean_runtime={r['Mean Duration (s)']:.4g}s, n={int(r['Runs'])}\n"
                )

    print("Saved:", csv_path)
    print("Saved:", txt_path)
    return df


df_stats = generate_statistical_report(recaps)
df_stats


### 2.1 Runtime and Function-Evaluation Summary


In [ ]:
efficiency_rows = []
for obj in OBJECTIVES:
    for algo in ALGORITHMS:
        durations = _finite_float_array(recaps[obj][algo]["durations"])
        fe = _finite_float_array(recaps[obj][algo]["function_evaluations"])
        if durations.size or fe.size:
            efficiency_rows.append({
                "Objective": obj.upper(),
                "Algorithm": algo_label(algo),
                "Mean Runtime (s)": float(np.mean(durations)) if durations.size else np.nan,
                "Std Runtime (s)": float(np.std(durations, ddof=1)) if durations.size > 1 else 0.0,
                "Mean FE": float(np.mean(fe)) if fe.size else np.nan,
                "Median FE": float(np.median(fe)) if fe.size else np.nan,
                "Runs": int(max(durations.size, fe.size)),
            })

df_efficiency = pd.DataFrame(efficiency_rows)
df_efficiency.to_csv(DATA_DIR / "statistics" / "computational_efficiency.csv", index=False)

if not df_efficiency.empty:
    pivot = df_efficiency.pivot(index="Objective", columns="Algorithm", values="Mean Runtime (s)")
    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title("Average optimization runtime")
    ax.set_xlabel("Objective function")
    ax.set_ylabel("Mean runtime (s)")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(DATA_DIR / "statistics" / "runtime_comparison.png", dpi=300)
    plt.close()

df_efficiency


## 3. Convergence Analysis

Best-so-far histories from all 30 runs are summarized pointwise using the median and interquartile range (IQR). This is the convergence view used by the current manuscript; per-run figure generation is intentionally omitted from the main analysis pipeline.


In [ ]:
def _valid_curves(obj, algo):
    curves = []
    for curve in recaps[obj][algo]["curves"]:
        if curve is None:
            continue
        arr = np.asarray(curve, dtype=float).reshape(-1)
        if arr.size:
            curves.append(arr)
    return curves


def plot_median_iqr_convergence():
    for obj in OBJECTIVES:
        fig, ax = plt.subplots(figsize=(8, 5))
        for algo in ALGORITHMS:
            curves = _valid_curves(obj, algo)
            if not curves:
                continue
            min_len = min(map(len, curves))
            X = np.vstack([curve[:min_len] for curve in curves])
            median = np.median(X, axis=0)
            q25 = np.percentile(X, 25, axis=0)
            q75 = np.percentile(X, 75, axis=0)
            iterations = np.arange(1, min_len + 1)
            line, = ax.plot(iterations, median, label=algo_label(algo))
            ax.fill_between(iterations, q25, q75, alpha=0.16, color=line.get_color())

        ax.set_title(f"{obj.upper()} — median convergence with IQR")
        ax.set_xlabel("Iteration")
        ax.set_ylabel("Best-so-far objective")
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(DATA_DIR / "convergence" / "summary" / f"{obj}_median_iqr.png", dpi=300)
        plt.close(fig)


plot_median_iqr_convergence()
print("Median/IQR convergence plots generated.")


## 4. Statistical Analysis


### 4.1 Q–Q Plot (Visual/Informal Normality Check)


In [ ]:
def qq_plot_one(data, title, outpath):
    x = _finite_float_array(data)
    if x.size < 3:
        return

    fig, ax = plt.subplots(figsize=(6, 6))
    (osm, osr), (slope, intercept, r) = stats.probplot(x, dist="norm")
    ax.scatter(osm, osr, s=20)
    xline = np.array([osm.min(), osm.max()])
    ax.plot(xline, slope * xline + intercept)
    ax.set_title(title)
    ax.set_xlabel("Theoretical quantiles")
    ax.set_ylabel("Ordered sample values")
    ax.grid(alpha=0.3)
    ax.text(0.05, 0.95, f"Fit r = {r:.4f}", transform=ax.transAxes, va="top")
    fig.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)


for obj in OBJECTIVES:
    for algo in ALGORITHMS:
        data = recaps[obj][algo]["best_fitness"]
        if data:
            qq_plot_one(
                data,
                f"Q–Q Plot: {obj.upper()} — {algo_label(algo)} (n={len(data)})",
                DATA_DIR / "statistics" / "normality" / "qqplots" / f"qq_{obj}_{algo}.png",
            )

print("Q–Q plots generated.")


### 4.2 Shapiro–Wilk Test


In [ ]:
shapiro_rows = []

for obj in OBJECTIVES:
    for algo in ALGORITHMS:
        x = _finite_float_array(recaps[obj][algo]["best_fitness"])
        if x.size < 3:
            continue

        if np.ptp(x) == 0:
            stat, p = np.nan, 1.0
            conclusion = "Degenerate"
        else:
            stat, p = shapiro(x)
            conclusion = "Normal" if p >= 0.05 else "Not normal"

        shapiro_rows.append({
            "Objective": obj.upper(),
            "Algorithm": algo_label(algo),
            "N": len(x),
            "W": stat,
            "p-value": p,
            "Conclusion": conclusion,
        })

df_shapiro = pd.DataFrame(shapiro_rows)
df_shapiro.to_csv(DATA_DIR / "statistics" / "normality" / "shapiro_results.csv", index=False)
df_shapiro


### 4.3 Friedman Test

Run dibandingkan berdasarkan **run ID yang sama** di semua algoritma. Ini menghindari pairing palsu akibat sekadar memotong array ke panjang minimum.


In [ ]:
def aligned_fitness_matrix(obj: str, algos: List[str]):
    maps = {
        algo: {
            rid: fit
            for rid, fit in zip(recaps[obj][algo]["run_ids"], recaps[obj][algo]["best_fitness"])
            if fit is not None and np.isfinite(float(fit))
        }
        for algo in algos
    }
    common_ids = sorted(set.intersection(*(set(m.keys()) for m in maps.values()))) if maps else []
    if not common_ids:
        return [], np.empty((0, len(algos)))
    X = np.array([[maps[a][rid] for a in algos] for rid in common_ids], dtype=float)
    return common_ids, X


friedman_rows = []
friedman_results = {}

for obj in OBJECTIVES:
    run_ids, X = aligned_fitness_matrix(obj, ALGORITHMS)
    if X.shape[0] < 2:
        continue

    chi2, p = friedmanchisquare(*[X[:, i] for i in range(X.shape[1])])
    ranks = np.apply_along_axis(lambda row: rankdata(row, method="average"), 1, X)
    mean_ranks = ranks.mean(axis=0)
    n_blocks, n_algorithms = X.shape
    kendall_w = float(chi2 / (n_blocks * (n_algorithms - 1)))

    friedman_results[obj] = {
        "run_ids": run_ids,
        "X": X,
        "chi2": chi2,
        "p": p,
        "kendall_w": kendall_w,
        "mean_ranks": dict(zip(ALGORITHMS, mean_ranks)),
    }

    for algo, mean_rank in zip(ALGORITHMS, mean_ranks):
        friedman_rows.append({
            "Objective": obj.upper(),
            "Algorithm": algo_label(algo),
            "PairedRuns": len(run_ids),
            "Mean Rank": mean_rank,
            "Chi-square": chi2,
            "Kendall W": kendall_w,
            "p-value": p,
            "Significant": p < 0.05,
        })

df_friedman = pd.DataFrame(friedman_rows)
df_friedman.to_csv(DATA_DIR / "statistics" / "comparative" / "friedman_results.csv", index=False)
df_friedman


### 4.4 Pairwise Wilcoxon + Bonferroni

All \(\binom{6}{2}=15\) algorithm pairs are compared two-sided using the same run IDs. Full-precision paired differences are retained; only exactly identical arrays are treated as a zero-difference case. Bonferroni correction is applied separately within each objective. Paired median differences and matched-pairs rank-biserial correlations are reported alongside significance.


In [ ]:
def matched_rank_biserial(x1, x2):
    diff = np.asarray(x1, dtype=float) - np.asarray(x2, dtype=float)
    diff = diff[diff != 0.0]
    if diff.size == 0:
        return 0.0
    ranks = rankdata(np.abs(diff), method="average")
    w_pos = float(np.sum(ranks[diff > 0.0]))
    w_neg = float(np.sum(ranks[diff < 0.0]))
    denom = w_pos + w_neg
    return (w_pos - w_neg) / denom if denom else 0.0


pairwise_rows = []
for obj in OBJECTIVES:
    for a1, a2 in itertools.combinations(ALGORITHMS, 2):
        maps = {
            algo: {
                rid: fit
                for rid, fit in zip(recaps[obj][algo]["run_ids"], recaps[obj][algo]["best_fitness"])
                if fit is not None and np.isfinite(float(fit))
            }
            for algo in (a1, a2)
        }
        common_ids = sorted(set(maps[a1]) & set(maps[a2]))
        if len(common_ids) < 2:
            continue

        x1 = np.array([maps[a1][rid] for rid in common_ids], dtype=float)
        x2 = np.array([maps[a2][rid] for rid in common_ids], dtype=float)
        diff = x1 - x2

        if np.array_equal(x1, x2):
            w_stat, p = 0.0, 1.0
        else:
            w_stat, p = wilcoxon(x1, x2, alternative="two-sided", zero_method="wilcox")

        pairwise_rows.append({
            "Objective": obj.upper(),
            "Algorithm A": algo_label(a1),
            "Algorithm B": algo_label(a2),
            "PairedRuns": len(common_ids),
            "W-stat": w_stat,
            "p-value": p,
            "Median Difference (A-B)": float(np.median(diff)),
            "Rank-biserial (A-B)": matched_rank_biserial(x1, x2),
        })


df_w = pd.DataFrame(pairwise_rows)
if not df_w.empty:
    corrected_parts = []
    for objective, part in df_w.groupby("Objective", sort=False):
        part = part.copy()
        m = len(part)
        part["Bonferroni alpha"] = 0.05 / m
        part["Adjusted p-value"] = np.minimum(part["p-value"] * m, 1.0)
        part["Significant (Bonferroni)"] = part["Adjusted p-value"] < 0.05
        corrected_parts.append(part)
    df_w = pd.concat(corrected_parts, ignore_index=True)

df_w.to_csv(DATA_DIR / "statistics" / "comparative" / "wilcoxon_pairwise_bonferroni.csv", index=False)
df_w


### 4.5 Boxplot Analysis


In [ ]:
plot_rows = []
for obj in OBJECTIVES:
    for algo in ALGORITHMS:
        for value in recaps[obj][algo]["best_fitness"]:
            if value is not None and np.isfinite(float(value)):
                plot_rows.append({
                    "Objective": obj.upper(),
                    "Algorithm": algo_label(algo),
                    "Fitness": float(value),
                })

df_plot = pd.DataFrame(plot_rows)

if not df_plot.empty:
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()

    for ax, obj in zip(axes, OBJECTIVES):
        subset = df_plot[df_plot["Objective"] == obj.upper()]
        series = [
            subset.loc[subset["Algorithm"] == algo_label(a), "Fitness"].to_numpy()
            for a in ALGORITHMS
        ]
        labels = [algo_label(a) for a in ALGORITHMS]
        valid = [(label, x) for label, x in zip(labels, series) if len(x)]

        if valid:
            ax.boxplot(
                [x for _, x in valid],
                tick_labels=[label for label, _ in valid],
                showfliers=True
            )
        ax.set_title(obj.upper())
        ax.set_ylabel("Final objective value")
        ax.tick_params(axis="x", rotation=20)
        ax.grid(axis="y", alpha=0.3)

    fig.suptitle("Final-fitness distributions across independent runs")
    fig.tight_layout()
    fig.savefig(DATA_DIR / "statistics" / "comparative" / "boxplot_comparison.png", dpi=300)
    plt.close(fig)

print("Boxplot generated.")


## 5. Median-Representative Gain Selection

Controller-level validation uses an **observed optimizer run whose final objective is closest to the 30-run median**. This avoids turning downstream validation into a showcase of the luckiest stochastic run while also avoiding synthetic averaging of gain vectors.


In [ ]:
def select_solution(bucket, method="median"):
    fitness = np.asarray(bucket["best_fitness"], dtype=float)
    solutions = bucket["best_solutions"]
    valid = np.isfinite(fitness)
    if not np.any(valid):
        return None

    valid_indices = np.where(valid)[0]
    valid_fitness = fitness[valid]

    if method == "median":
        target = np.median(valid_fitness)
        local_idx = int(np.argmin(np.abs(valid_fitness - target)))
        idx = int(valid_indices[local_idx])
    elif method == "best":
        idx = int(valid_indices[np.argmin(valid_fitness)])
    else:
        raise ValueError(f"Unknown selection method: {method}")

    solution = np.asarray(solutions[idx], dtype=float).reshape(-1)
    if len(solution) != len(GAIN_NAMES):
        raise ValueError(f"Expected {len(GAIN_NAMES)} gains {GAIN_NAMES}, got {solution}.")
    return solution.tolist()


params = {
    obj: {
        algo: select_solution(recaps[obj][algo], method=PARAM_SELECTION)
        for algo in ALGORITHMS
        if recaps[obj][algo]["best_fitness"]
    }
    for obj in OBJECTIVES
}
params = {
    obj: {algo: p for algo, p in algos.items() if p is not None}
    for obj, algos in params.items()
}
params


## 6. Closed-Loop Utilities


In [ ]:
from benchmarks.sisken_prastiyanto import lean_simulate_system as acc_simulate
from core.fast_acc import (
    OBJECTIVE_IAE,
    OBJECTIVE_ISE,
    OBJECTIVE_ITAE,
    OBJECTIVE_ITSE,
    fast_acc_validation_metrics,
    fast_cruise_history,
)
from core.utils import (
    compute_jerk,
    compute_ttc_series,
    iae,
    ise,
    itae,
    itse,
    step_response_metrics,
)

OBJECTIVE_FUNCS = {
    "iae": iae,
    "ise": ise,
    "itae": itae,
    "itse": itse,
}

OBJECTIVE_IDS = {
    "iae": OBJECTIVE_IAE,
    "ise": OBJECTIVE_ISE,
    "itae": OBJECTIVE_ITAE,
    "itse": OBJECTIVE_ITSE,
}


def finite_min(x):
    x = np.asarray(x, dtype=float)
    finite = x[np.isfinite(x)]
    return float(np.min(finite)) if finite.size else math.inf


## 7. Time Response — Cruise Control


In [ ]:
def simulate_cruise_control(
    gains,
    sim_time=60.0,
    theta_angle=0.0,
    vehicle_mass=VEHICLE_MASS,
    rolling_resistance=C_R_NOMINAL,
    v_ref=25.0,
    v_ego_init=22.0,
):
    """Compiled pure speed-branch validation using the active vehicle model."""
    return fast_cruise_history(
        gains,
        sim_time=sim_time,
        theta_angle=theta_angle,
        vehicle_mass=vehicle_mass,
        rolling_resistance=rolling_resistance,
        v_ref=v_ref,
        v_ego_init=v_ego_init,
    )


cruise_metric_rows = []

for obj, algo_params in params.items():
    fig, ax = plt.subplots(figsize=(9, 5))

    for algo, gains in algo_params.items():
        hist = simulate_cruise_control(gains)
        metrics = step_response_metrics(hist["time"], hist["v_ego"], ref=25.0)

        cruise_metric_rows.append({
            "Objective": obj.upper(),
            "Algorithm": algo_label(algo),
            **metrics,
        })

        ax.plot(hist["time"], hist["v_ego"], label=algo_label(algo))

    ax.axhline(25.0, linestyle="--", label="Reference")
    ax.set_title(f"Cruise-control speed response — {obj.upper()}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Ego speed (m/s)")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(DATA_DIR / "time_response" / "cruise_control" / f"speed_{obj}.png", dpi=300)
    plt.close(fig)

df_cruise_metrics = pd.DataFrame(cruise_metric_rows)
df_cruise_metrics.to_csv(DATA_DIR / "time_response" / "cruise_control" / "metrics.csv", index=False)
df_cruise_metrics


## 8. TTC-based Car Following


In [ ]:
def run_car_following(gains, objective_name, **scenario):
    obj_fun = OBJECTIVE_FUNCS[objective_name]
    sensor_cfg = {**NOMINAL_SENSOR, **{
        k: scenario[k]
        for k in NOMINAL_SENSOR
        if k in scenario
    }}
    cost, history = acc_simulate(
        gains,
        obj_fun,
        return_history=True,
        theta_angle=scenario.get("theta_angle", 0.0),
        vehicle_mass=scenario.get("vehicle_mass", VEHICLE_MASS),
        rolling_resistance=scenario.get("rolling_resistance", C_R_NOMINAL),
        **sensor_cfg,
    )

    history = dict(history)
    history["ttc"] = compute_ttc_series(
        history["dist"], history["v_ego"], history["v_lead"]
    )
    history["jerk"] = compute_jerk(history["a_ego"], DT)
    return float(cost), history


car_follow_rows = []
for obj, algo_params in params.items():
    histories = {}
    for algo, gains in algo_params.items():
        cost, hist = run_car_following(gains, obj)
        histories[algo] = hist
        car_follow_rows.append({
            "Objective": obj.upper(),
            "Algorithm": algo_label(algo),
            "ObjectiveValue": cost,
            "TTC_Min": finite_min(hist["ttc"]),
            "Max_Jerk": float(np.max(np.abs(hist["jerk"]))),
            "Min_Distance": float(np.min(hist["dist"])),
        })

    if not histories:
        continue

    fig, ax = plt.subplots(figsize=(9, 5))
    for algo, hist in histories.items():
        ttc = np.asarray(hist["ttc"], dtype=float)
        ttc_plot = np.where(np.isfinite(ttc), np.minimum(ttc, 20.0), 20.0)
        ax.plot(hist["time"], ttc_plot, label=algo_label(algo))
    ax.axhline(TTC_THRESHOLD, linestyle="--", label=f"TTC threshold = {TTC_THRESHOLD}s")
    ax.set_title(f"TTC comparison — {obj.upper()}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("TTC (s), capped at 20 s")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(DATA_DIR / "time_response" / "car_following" / "ttc" / f"ttc_{obj}.png", dpi=300)
    plt.close(fig)

    fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
    any_hist = next(iter(histories.values()))
    axes[1].plot(any_hist["time"], any_hist["v_lead"], linestyle="--", label="Lead")
    axes[2].plot(any_hist["time"], any_hist["a_lead"], linestyle="--", label="Lead")
    for algo, hist in histories.items():
        axes[0].plot(hist["time"], hist["dist"], label=algo_label(algo))
        axes[0].plot(hist["time"], hist["d_safe"], linestyle=":", alpha=0.5)
        axes[1].plot(hist["time"], hist["v_ego"], label=algo_label(algo))
        axes[2].plot(hist["time"], hist["a_ego"], label=algo_label(algo))
    axes[0].set_ylabel("Distance (m)")
    axes[0].set_title(f"Car-following motion profile — {obj.upper()}")
    axes[1].set_ylabel("Velocity (m/s)")
    axes[2].set_ylabel("Acceleration (m/s²)")
    axes[2].set_xlabel("Time (s)")
    for ax in axes:
        ax.grid(alpha=0.3)
    axes[0].legend()
    axes[1].legend()
    axes[2].legend()
    fig.tight_layout()
    fig.savefig(DATA_DIR / "time_response" / "car_following" / "motion" / f"motion_{obj}.png", dpi=300)
    plt.close(fig)


df_car_follow = pd.DataFrame(car_follow_rows)
df_car_follow.to_csv(DATA_DIR / "time_response" / "car_following" / "metrics.csv", index=False)
df_car_follow


## 9. Sensing Diagnostic

The simulator now preserves an explicit sensor boundary even in the nominal zero-noise case. These plots compare simulator-side separation with the controller-visible range and compare finite-difference range rate with evaluator-side ground truth. The same `run_car_following` helper accepts `sensor_noise_std_m`, `sensor_bias_m`, and `sensor_seed` for later sensing-robustness experiments.


In [ ]:
for obj, algo_params in params.items():
    if not algo_params:
        continue

    first_algo = next(iter(algo_params))
    _, hist = run_car_following(algo_params[first_algo], obj)
    t = np.asarray(hist["time"])

    fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True)
    axes[0].plot(t, hist["dist"], label="Ground-truth separation")
    axes[0].plot(t, hist["dist_sensed"], linestyle="--", label="Sensed range")
    axes[0].set_ylabel("Range (m)")
    axes[0].set_title(f"Sensing diagnostic — {obj.upper()} / {algo_label(first_algo)}")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(t, hist["v_rel_est"], label="Estimated range rate (controller input)")
    axes[1].plot(t, hist["v_rel_true"], linestyle="--", label="Ground truth (evaluator only)")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Relative velocity (m/s)")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(
        DATA_DIR / "time_response" / "car_following" / "sensing" / f"sensing_{obj}.png",
        dpi=300,
    )
    plt.close(fig)

print("Sensing diagnostic plots generated.")


## 10. Physical-Parameter Robustness

The existing 12 modeled plant variations change mass, rolling resistance, and road slope while keeping the selected controller gains fixed. Sensor perturbations are supported by the simulator but are intentionally not assigned arbitrary noise levels here; any sensing-robustness experiment should define its noise/bias scenarios explicitly before execution.


In [ ]:
SCENARIOS = [
    {"id": "S0",  "desc": "Normal",        "theta_angle": 0,   "vehicle_mass": VEHICLE_MASS,        "rolling_resistance": C_R_NOMINAL},
    {"id": "S1",  "desc": "Mass +10%",     "theta_angle": 0,   "vehicle_mass": VEHICLE_MASS * 1.10, "rolling_resistance": C_R_NOMINAL},
    {"id": "S2",  "desc": "Mass -10%",     "theta_angle": 0,   "vehicle_mass": VEHICLE_MASS * 0.90, "rolling_resistance": C_R_NOMINAL},
    {"id": "S3",  "desc": "c_r +20%",      "theta_angle": 0,   "vehicle_mass": VEHICLE_MASS,        "rolling_resistance": C_R_PLUS_20},
    {"id": "S4",  "desc": "Up Normal",     "theta_angle": 10,  "vehicle_mass": VEHICLE_MASS,        "rolling_resistance": C_R_NOMINAL},
    {"id": "S5",  "desc": "Up Mass +10%",  "theta_angle": 10,  "vehicle_mass": VEHICLE_MASS * 1.10, "rolling_resistance": C_R_NOMINAL},
    {"id": "S6",  "desc": "Up Mass -10%",  "theta_angle": 10,  "vehicle_mass": VEHICLE_MASS * 0.90, "rolling_resistance": C_R_NOMINAL},
    {"id": "S7",  "desc": "Up c_r +20%",   "theta_angle": 10,  "vehicle_mass": VEHICLE_MASS,        "rolling_resistance": C_R_PLUS_20},
    {"id": "S8",  "desc": "Down Normal",   "theta_angle": -10, "vehicle_mass": VEHICLE_MASS,        "rolling_resistance": C_R_NOMINAL},
    {"id": "S9",  "desc": "Down Mass +10%", "theta_angle": -10, "vehicle_mass": VEHICLE_MASS * 1.10, "rolling_resistance": C_R_NOMINAL},
    {"id": "S10", "desc": "Down Mass -10%", "theta_angle": -10, "vehicle_mass": VEHICLE_MASS * 0.90, "rolling_resistance": C_R_NOMINAL},
    {"id": "S11", "desc": "Down c_r +20%",  "theta_angle": -10, "vehicle_mass": VEHICLE_MASS,        "rolling_resistance": C_R_PLUS_20},
]

pd.DataFrame(SCENARIOS)


### 10.1 Car-Following Robustness


In [ ]:
robust_cf_rows = []

for obj, algo_params in params.items():
    for scenario in SCENARIOS:
        for algo, gains in algo_params.items():
            cost, ttc_min, max_jerk, min_distance = fast_acc_validation_metrics(
                gains,
                OBJECTIVE_IDS[obj],
                theta_angle=scenario["theta_angle"],
                vehicle_mass=scenario["vehicle_mass"],
                rolling_resistance=scenario["rolling_resistance"],
                **NOMINAL_SENSOR,
            )
            robust_cf_rows.append({
                "Objective": obj.upper(),
                "Algorithm": algo_label(algo),
                "ScenarioID": scenario["id"],
                "Description": scenario["desc"],
                "ObjectiveValue": cost,
                "TTC_Min": ttc_min,
                "Max_Jerk": max_jerk,
                "Min_Distance": min_distance,
            })

df_robust_cf = pd.DataFrame(robust_cf_rows)
df_robust_cf.to_csv(
    DATA_DIR / "robustness" / "car_following" / "results" / "car_following_metrics.csv",
    index=False,
)
df_robust_cf.head()


### 10.2 Cruise-Control Robustness


In [ ]:
robust_cruise_rows = []

for obj, algo_params in params.items():
    for scenario in SCENARIOS:
        for algo, gains in algo_params.items():
            hist = simulate_cruise_control(
                gains,
                theta_angle=scenario["theta_angle"],
                vehicle_mass=scenario["vehicle_mass"],
                rolling_resistance=scenario["rolling_resistance"],
            )
            metrics = step_response_metrics(hist["time"], hist["v_ego"], ref=25.0)

            robust_cruise_rows.append({
                "Objective": obj.upper(),
                "Algorithm": algo_label(algo),
                "ScenarioID": scenario["id"],
                "Description": scenario["desc"],
                "RiseTime": metrics["rise_time"],
                "SettlingTime": metrics["settling_time"],
                "OvershootPct": metrics["overshoot_pct"],
                "SteadyStateError": metrics["steady_state_error"],
                "PeakTime": metrics["peak_time"],
            })

df_robust_cruise = pd.DataFrame(robust_cruise_rows)
df_robust_cruise.to_csv(
    DATA_DIR / "robustness" / "cruise_control" / "results" / "cruise_control_robustness_report.csv",
    index=False,
)
df_robust_cruise.head()


### 10.3 Robustness Metric Charts


In [ ]:
def plot_metric(df, metric, ylabel, out_dir, prefix):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for obj in OBJECTIVES:
        subset = df[df["Objective"] == obj.upper()]
        if subset.empty or metric not in subset:
            continue

        pivot = subset.pivot(index="Description", columns="Algorithm", values=metric)
        ax = pivot.plot(kind="bar", figsize=(12, 5))
        ax.set_title(f"{ylabel} — {obj.upper()}")
        ax.set_xlabel("Scenario")
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(out_dir / f"{prefix}_{obj}_{metric.lower()}.png", dpi=250)
        plt.close()


if not df_robust_cruise.empty:
    plot_metric(
        df_robust_cruise, "SettlingTime", "Settling time (s)",
        DATA_DIR / "robustness" / "cruise_control" / "plots", "cruise"
    )
    plot_metric(
        df_robust_cruise, "SteadyStateError", "Steady-state error (m/s)",
        DATA_DIR / "robustness" / "cruise_control" / "plots", "cruise"
    )

if not df_robust_cf.empty:
    plot_metric(
        df_robust_cf, "TTC_Min", "Minimum TTC (s)",
        DATA_DIR / "robustness" / "car_following" / "plots", "following"
    )
    plot_metric(
        df_robust_cf, "Max_Jerk", "Maximum jerk (m/s³)",
        DATA_DIR / "robustness" / "car_following" / "plots", "following"
    )
    plot_metric(
        df_robust_cf, "ObjectiveValue", "Objective value",
        DATA_DIR / "robustness" / "car_following" / "plots", "following"
    )

print("Robustness metric plots generated.")


## 11. Final Consistency Checks

The final cell validates the archived campaign against the versioned manifest: coverage, curve length, gain dimensionality/bounds, and availability of function-evaluation accounting.


In [ ]:
issues = []

for obj in OBJECTIVES:
    for algo in ALGORITHMS:
        bucket = recaps[obj][algo]
        n = len(bucket["run_ids"])
        if n != EXPECTED_RUNS:
            issues.append(f"{obj.upper()} / {algo_label(algo)}: {n}/{EXPECTED_RUNS} runs.")

        for run_id, curve in zip(bucket["run_ids"], bucket["curves"]):
            if curve is None:
                issues.append(f"{obj.upper()} / {algo_label(algo)} / run {run_id}: missing convergence curve.")
                continue
            curve_len = len(np.asarray(curve).reshape(-1))
            if curve_len != EXPECTED_ITERATIONS:
                issues.append(
                    f"{obj.upper()} / {algo_label(algo)} / run {run_id}: "
                    f"{curve_len}/{EXPECTED_ITERATIONS} iterations."
                )

        for run_id, fe in zip(bucket["run_ids"], bucket["function_evaluations"]):
            if fe is None or not np.isfinite(float(fe)) or float(fe) <= 0:
                issues.append(f"{obj.upper()} / {algo_label(algo)} / run {run_id}: missing/invalid FE count.")

for obj, algo_params in params.items():
    for algo, gains in algo_params.items():
        gains = np.asarray(gains, dtype=float)
        if gains.size != len(GAIN_NAMES):
            issues.append(f"{obj}/{algo}: expected {len(GAIN_NAMES)} gains, got {gains}.")
            continue
        if np.any(gains < MIN_PARAMS) or np.any(gains > MAX_PARAMS):
            issues.append(f"{obj}/{algo}: representative gains outside manifest bounds: {gains}.")

expected_total = EXPECTED_RUNS * len(ALGORITHMS) * len(OBJECTIVES)
if len(checkpoint_rows) != expected_total:
    issues.append(f"Checkpoint total {len(checkpoint_rows)}/{expected_total}.")

print(f"Benchmark key: {BENCHMARK_KEY}")
print(f"Gain semantics: {GAIN_NAMES}")
print(f"Gain bounds: {MIN_PARAMS.tolist()} to {MAX_PARAMS.tolist()}")
print(f"Algorithms: {[algo_label(a) for a in ALGORITHMS]}")
print(f"Expected runs / iterations / population: {EXPECTED_RUNS} / {EXPECTED_ITERATIONS} / {POPULATION_SIZE}")
print(f"Representative-run policy: {PARAM_SELECTION}")
print(f"Nominal sensor: {NOMINAL_SENSOR}")

if issues:
    print("\n[CHECK] Items requiring attention:")
    for item in issues:
        print(" -", item)
else:
    print("\n[CHECK] Archived campaign is consistent with the active manifest.")
